In [ ]:
import boto3
import os
from boto3.s3.transfer import TransferConfig
import threading
import sys
import json

access_key_id = 'nope'
secret_access_key = 'nope'
LOCAL_S3_PROXY_SERVICE_URL = 'https://nope.nope.com'

s3 = boto3.client('s3',
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    endpoint_url=LOCAL_S3_PROXY_SERVICE_URL
)

In [6]:
response = s3.create_bucket(
    Bucket='this-is-2'
)

BucketAlreadyOwnedByYou: An error occurred (BucketAlreadyOwnedByYou) when calling the CreateBucket operation: Your previous request to create the named bucket succeeded and you already own it.

In [7]:
s3.list_buckets()['Buckets']

[{'Name': 'this-is-2',
  'CreationDate': datetime.datetime(2025, 8, 20, 6, 20, 31, 947000, tzinfo=tzutc())}]

In [8]:
s3.list_objects_v2(Bucket='this-is-2')['Contents']

[{'Key': 'PS2_BIOS.zip',
  'LastModified': datetime.datetime(2025, 8, 20, 6, 23, 8, 113000, tzinfo=tzutc()),
  'ETag': '"d1ad2e2cac1656c266266dac9b105470-35"',
  'Size': 181087479,
  'StorageClass': 'STANDARD'}]

In [ ]:
config = TransferConfig(
    multipart_threshold=1024 * 25,  # 25MB
    max_concurrency=10,
    multipart_chunksize=1024 * 25,
    use_threads=True
)

class ProgressPercentage:
    def __init__(self, filename):
        self._filename = filename
        self._size = float(os.path.getsize(filename))
        self._seen_so_far = 0
        self._lock = threading.Lock()

    def __call__(self, bytes_amount):
        with self._lock:
            self._seen_so_far += bytes_amount
            percentage = (self._seen_so_far / self._size) * 100
            sys.stdout.write(
                f"\r{self._filename}: {self._seen_so_far} / {self._size} "
                f"({percentage:.2f}%)"
            )
            sys.stdout.flush()

# Upload with progress
s3.upload_file(
    '/Users/nope/Downloads/ProtonVPN_mac_v5.1.0.dmg',
    'this-is-2',
    'ProtonVPN_mac_v5.1.0.dmg',
    Config=config,
    Callback=ProgressPercentage('/Users/nope/Downloads/ProtonVPN_mac_v5.1.0.dmg')
)

/Users/svo/Downloads/ProtonVPN_mac_v5.1.0.dmg: 122895063 / 122895063.0 (100.00%)

In [ ]:
s3.download_file(
    'this-is-2',
    'ProtonVPN_mac_v5.1.0.dmg',
    '/Users/nope/Downloads/ProtonVPN_mac_vREDOWNLOADED.dmg',
)